In [1]:
import pandas as pd
FILE_PATH = '/kaggle/input/datasets/shaswatshrey/processed-data/final_text_pairs.tsv'

processed_df = pd.read_csv(FILE_PATH, sep='\t')
print(processed_df.head(), flush=True)
length = len(processed_df)
print("length of the dataset:", length)

count = (processed_df['label'] == 1).sum()
print("Number of rows with label 1:", count)
print("Number of rows with label 0:", length-count)

     source1_id  candidate_id  label  \
0  S1-925783039  S2-335180728      0   
1  S1-925783039  S2-544837895      0   
2  S1-925783039  S2-490045451      0   
3  S1-925783039  S2-157377754      1   
4  S1-925783039  S2-668494204      0   

                                             text_s1  \
0  Name: Orelee's Barbershop | Address: 1795 West...   
1  Name: Orelee's Barbershop | Address: 1795 West...   
2  Name: Orelee's Barbershop | Address: 1795 West...   
3  Name: Orelee's Barbershop | Address: 1795 West...   
4  Name: Orelee's Barbershop | Address: 1795 West...   

                                           text_cand  
0  Name: Orelee's Barbershop Downtown | Address: ...  
1  Name: Orelee's Group | Address: 14 CHEROKEE AV...  
2  Name: ORELEE'S DIRECT PÁRTNERS | Address: 1377...  
3  Name: Orelee's Barbershop | Address: 1795 WEST...  
4  Name: Orelee's Dístribution | Address: BAHAMA,...  
length of the dataset: 38950247
Number of rows with label 1: 1570475
Number of rows with lab

In [2]:
import pandas as pd
import hashlib

INPUT_PATH = '/kaggle/input/datasets/shaswatshrey/processed-data/final_text_pairs.tsv'

TRAIN_PATH = '/kaggle/working/train_pairs.tsv'
VAL_PATH = '/kaggle/working/val_pairs.tsv'
TEST_PATH = '/kaggle/working/test_pairs.tsv'

CHUNK_SIZE = 200_000

def assign_split(source_id):
    source_id = str(source_id)

    hash_value = hashlib.md5(
        source_id.encode('utf-8')
    ).hexdigest()

    bucket = int(hash_value, 16) % 100

    if bucket < 80:
        return "train"
    elif bucket < 90:
        return "val"
    else:
        return "test"
    

In [3]:
first_chunk = True

for chunk in pd.read_csv(INPUT_PATH, sep='\t', chunksize=CHUNK_SIZE):
    chunk["split"] = chunk['source1_id'].apply(assign_split)

    # Separate
    train_chunk = chunk[chunk["split"] == "train"]
    val_chunk   = chunk[chunk["split"] == "val"]
    test_chunk  = chunk[chunk["split"] == "test"]

    # Save
    train_chunk.drop(columns=["split"]).to_csv(
        TRAIN_PATH,
        sep="\t",
        index=False,
        mode="w" if first_chunk else "a",
        header=first_chunk
    )

    val_chunk.drop(columns=["split"]).to_csv(
        VAL_PATH,
        sep="\t",
        index=False,
        mode="w" if first_chunk else "a",
        header=first_chunk
    )

    test_chunk.drop(columns=["split"]).to_csv(
        TEST_PATH,
        sep="\t",
        index=False,
        mode="w" if first_chunk else "a",
        header=first_chunk
    )

    first_chunk = False

    print(
        f"Processed {len(chunk):,} rows"
    )
    

Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 200,000 rows
Processed 2

In [4]:
import pandas as pd

for name, PATH in [
    ("TRAIN", TRAIN_PATH),
    ("VALIDATION", VAL_PATH),
    ("TEST", TEST_PATH)
]:
    print(f"======={name}=======")

    total = 0
    positives = 0
    negatives = 0

    for chunk in pd.read_csv(
        PATH,
        sep = "\t",
        chunksize = CHUNK_SIZE
    ):
        total += len(chunk)
        positives += (chunk["label"] == 1).sum()
        negatives += (chunk["label"] == 0).sum()

    print("Rows:", f"{total:,}")
    print("Positive:", f"{positives:,}")
    print("Negative:", f"{negatives:,}")

=======TRAIN=======
Rows: 31,136,165
Positive: 1,256,346
Negative: 29,879,819
=======VALIDATION=======
Rows: 3,913,592
Positive: 158,034
Negative: 3,755,558
=======TEST=======
Rows: 3,900,490
Positive: 156,095
Negative: 3,744,395


In [5]:
import pandas as pd

TRAIN_PATH = "/kaggle/working/train_pairs.tsv"
OUTPUT_PATH = "/kaggle/working/sbert_train_600k.tsv"

TARGET_POSITIVE = 300_000
TARGET_NEGATIVE = 300_000

CHUNK_SIZE = 200_000

positive_parts = []
negative_parts = []

positive_count = 0
negative_count = 0

for chunk in pd.read_csv(
    TRAIN_PATH,
    sep="\t",
    chunksize=CHUNK_SIZE
):
    positive_rows = chunk[chunk["label"] == 1]
    negative_rows = chunk[chunk["label"] == 0]
    
    positive_needed = (TARGET_POSITIVE - positive_count)
    negative_needed = (TARGET_NEGATIVE - negative_count)

    if positive_needed > 0 and len(positive_rows) > 0:
        take = min(positive_needed, len(positive_rows))
        sampled_positive = positive_rows.sample(
            n=take,
            random_state=42
        )
        positive_parts.append(sampled_positive)
        positive_count += len(sampled_positive)

    if negative_needed > 0 and len(negative_rows) > 0:
        take = min(negative_needed,len(negative_rows))
        sampled_negative = negative_rows.sample(
            n=take,
            random_state=42
        )
        negative_parts.append(sampled_negative)

        negative_count += len(sampled_negative)

    print(
        f"Positive: "
        f"{positive_count:,}/{TARGET_POSITIVE:,} | "
        f"Negative: "
        f"{negative_count:,}/{TARGET_NEGATIVE:,}"
    )

    if (
        positive_count >= TARGET_POSITIVE
        and
        negative_count >= TARGET_NEGATIVE
    ):
        break


Positive: 9,610/300,000 | Negative: 190,390/300,000
Positive: 18,909/300,000 | Negative: 300,000/300,000
Positive: 28,544/300,000 | Negative: 300,000/300,000
Positive: 37,789/300,000 | Negative: 300,000/300,000
Positive: 47,099/300,000 | Negative: 300,000/300,000
Positive: 56,146/300,000 | Negative: 300,000/300,000
Positive: 65,367/300,000 | Negative: 300,000/300,000
Positive: 74,581/300,000 | Negative: 300,000/300,000
Positive: 83,805/300,000 | Negative: 300,000/300,000
Positive: 93,286/300,000 | Negative: 300,000/300,000
Positive: 102,784/300,000 | Negative: 300,000/300,000
Positive: 112,064/300,000 | Negative: 300,000/300,000
Positive: 121,428/300,000 | Negative: 300,000/300,000
Positive: 130,595/300,000 | Negative: 300,000/300,000
Positive: 139,948/300,000 | Negative: 300,000/300,000
Positive: 149,166/300,000 | Negative: 300,000/300,000
Positive: 158,787/300,000 | Negative: 300,000/300,000
Positive: 168,200/300,000 | Negative: 300,000/300,000
Positive: 177,450/300,000 | Negative: 3

In [6]:
positive_df = pd.concat(
    positive_parts,
    ignore_index=True
)
negative_df = pd.concat(
    negative_parts,
    ignore_index=True
)
# Combine positive + negative
sbert_train = pd.concat(
    [positive_df, negative_df],
    ignore_index=True
)
# Shuffle the complete dataset
sbert_train = sbert_train.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)
# Save
sbert_train.to_csv(
    OUTPUT_PATH,
    sep="\t",
    index=False
)



print("\n==============================")
print("FINAL SBERT TRAINING DATA")
print("==============================")

print("Shape:", sbert_train.shape)

print("\nLabel distribution:")
print(sbert_train["label"].value_counts())

print("\nSaved to:")
print(OUTPUT_PATH)


FINAL SBERT TRAINING DATA
Shape: (600000, 5)

Label distribution:
label
1    300000
0    300000
Name: count, dtype: int64

Saved to:
/kaggle/working/sbert_train_600k.tsv


In [7]:
!pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 26.8 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.4.1
    Uninstalling sentence-transformers-5.4.1:
      Successfully uninstalled sentence-transformers-5.4.1


In [8]:
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer
from sentence_transformers import InputExample
from sentence_transformers import losses

from torch.utils.data import DataLoader


/tmp/ipykernel_23/1016635433.py:6: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import losses


In [9]:

TRAIN_PATH = "/kaggle/working/sbert_train_600k.tsv"

MODEL_NAME = "all-MiniLM-L6-v2"

OUTPUT_MODEL_PATH = "/kaggle/working/sbert_entity_matching_v1"

BATCH_SIZE = 32
EPOCHS = 5

In [10]:
print("=" * 60)
print("DEVICE CHECK")
print("=" * 60)

if torch.cuda.is_available():
    device = "cuda"
    print("GPU available!")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("WARNING: GPU not available. Training will be slow.")

print("Device:", device)


DEVICE CHECK
GPU available!
GPU: Tesla T4
Device: cuda


In [11]:
print("\n" + "=" * 60)
print("LOADING TRAINING DATA")
print("=" * 60)

df = pd.read_csv(
    TRAIN_PATH,
    sep="\t",
    usecols=[
        "text_s1",
        "text_cand",
        "label"
    ]
)

print("Dataset shape:", df.shape)

print("\nLabel distribution:")
print(df["label"].value_counts())

print("\nMissing values:")
print(df.isna().sum())

df["text_s1"] = df["text_s1"].astype(str)
df["text_cand"] = df["text_cand"].astype(str)
df["label"] = df["label"].astype(float)

print("Shape after cleaning:", df.shape)

print("\n" + "=" * 60)
print("LOADING SBERT MODEL")
print("=" * 60)

model = SentenceTransformer(
    MODEL_NAME,
    device=device
)

print("Model loaded:", MODEL_NAME)


LOADING TRAINING DATA
Dataset shape: (600000, 3)

Label distribution:
label
1    300000
0    300000
Name: count, dtype: int64

Missing values:
label        0
text_s1      0
text_cand    0
dtype: int64
Shape after cleaning: (600000, 3)

LOADING SBERT MODEL


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded: all-MiniLM-L6-v2


In [12]:
train_examples = [
    InputExample(
        texts=[
            row.text_s1,
            row.text_cand
        ],
        label=float(row.label)
    )
    for row in df.itertuples(index=False)
]

print("Training examples:", len(train_examples))

print("\n" + "=" * 60)
print("CREATING DATALOADER")
print("=" * 60)

train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=BATCH_SIZE
)

print("Batch size:", BATCH_SIZE)
print("Number of batches:", len(train_dataloader))

print("\n" + "=" * 60)
print("SETTING UP LOSS")
print("=" * 60)

train_loss = losses.ContrastiveLoss(
    model=model
)

print("Loss: ContrastiveLoss")


# ------------------------------------------------------------
# 10. CALCULATE WARMUP STEPS
# ------------------------------------------------------------

warmup_steps = int(
    len(train_dataloader) * 0.1
)

print("Warmup steps:", warmup_steps)

print("\n" + "=" * 60)
print("STARTING SBERT TRAINING")
print("=" * 60)

print("Epochs:", EPOCHS)
print("Training examples:", len(train_examples))
print("Batch size:", BATCH_SIZE)

model.fit(
    train_objectives=[
        (train_dataloader, train_loss)
    ],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    show_progress_bar=True
)

print("\nTraining completed!")


# ------------------------------------------------------------
# 12. SAVE FINE-TUNED MODEL
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SAVING MODEL")
print("=" * 60)

model.save(
    OUTPUT_MODEL_PATH
)
print(OUT_MODEL_PATH)

Training examples: 600000

CREATING DATALOADER
Batch size: 32
Number of batches: 18750

SETTING UP LOSS
Loss: ContrastiveLoss
Warmup steps: 1875

STARTING SBERT TRAINING
Epochs: 5
Training examples: 600000
Batch size: 32


Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.009194
1000,0.006515
1500,0.005753
2000,0.005113
2500,0.004561
3000,0.004305
3500,0.004168
4000,0.003918
4500,0.003754
5000,0.003644



Training completed!

SAVING MODEL


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

NameError: name 'OUT_MODEL_PATH' is not defined